# 10 · Fit each guide's effect on each gene — ordinary least squares

An alternative to `10_FitGuideEffects_NegativeBinomial`. **Run one of the two**,
then point `par_guide_lm_model` at whichever you ran; notebook 12 reads that
one.

Ordinary least squares per response gene on the log-normalised matrix, with
every knockout guide as a covariate plus `n_genes`, `mt_frac` and the Leiden
cluster as dummies. Faster than the negative binomial fit and fits all guides
at once rather than in guide blocks, but it models log-normalised values rather
than counts, so it does not respect the mean-variance relationship of the data.

**Reads** `par_save_filename_7`.
**Writes** coefficient tables into `par_guide_lm_dir/OLS`.

The original analysis wrote one wide matrix per gene block, covariates by
genes. This writes the same tidy layout as the negative binomial notebook —
one row per term per gene — so that notebook 12 reads either without changes.

:::{note}
The fits below take days to run across all guides, so their results are
provided rather than regenerated. `TextFiles/GuideSelect_BadKOGuides.csv` and
`TextFiles/GuideSelect_GoodGuides.csv` hold the guide selection the published
analysis used, and notebook 13 reads the first of them directly.

This notebook records how that selection was made. It writes its own results to
separate files under `outputs/` and leaves the provided lists untouched.
:::


## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
import statsmodels.api as sm

FIT_DIR = par_guide_lm_dir + "/OLS"
Path(FIT_DIR).mkdir(parents=True, exist_ok=True)

## Build the design matrix

The Leiden cluster enters as dummy columns rather than as a factor, so that the
fit is a plain linear model.

In [ ]:
adata = sc.read(par_save_filename_7)
print(f"input: {adata.shape[0]} cells x {adata.shape[1]} genes")

ko_guides = list(adata.uns["feature_KO_barcode_names_filtered"])

covariates = adata.obs[["n_genes", "mt_frac", "leiden"]]
covariates = covariates.join(pd.get_dummies(covariates.leiden)).drop(columns=["leiden"])

design = adata.obs[ko_guides].join(covariates).astype(float)
design_matrix = np.array(design)
terms = list(design.columns)

expression = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
gene_names = list(adata.var_names)
print(f"design: {design.shape[0]} cells x {design.shape[1]} covariates "
      f"({len(ko_guides)} guides)")

## Fit

One model per gene, in blocks of `par_gene_block_size`. Each block is written
as it completes and blocks already on disk are skipped, so an interrupted run
resumes where it stopped.

In [ ]:
n_genes = len(gene_names)

for start in range(0, n_genes, par_gene_block_size):
    stop = min(start + par_gene_block_size, n_genes)
    out = f"{FIT_DIR}/coefs_0_{len(ko_guides)}_{start}_{stop}.csv"
    if Path(out).exists():
        continue

    rows = []
    for j in range(start, stop):
        fit = sm.OLS(expression[:, j], design_matrix).fit()
        rows.append(pd.DataFrame({
            "term": terms,
            "estimate": fit.params,
            "std.error": fit.bse,
            "statistic": fit.tvalues,
            "p.value": fit.pvalues,
            "respGene": gene_names[j],
        }))

    pd.concat(rows, ignore_index=True).to_csv(out, index=False)
    print(f"  genes {start}-{stop} done")

print("all blocks fitted")

## Check

Notebook 12 drops the intercept and the QC covariates and pivots the rest into
a guide by gene matrix. The counts below are what it will see.

In [ ]:
files = sorted(Path(FIT_DIR).glob("coefs_*.csv"))
tidy = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

guide_terms = tidy[tidy.term.isin(ko_guides)]
print(f"blocks written : {len(files)}")
print(f"rows           : {len(tidy)}")
print(f"guide terms    : {guide_terms.term.nunique()} of {len(ko_guides)}")
print(f"response genes : {tidy.respGene.nunique()} of {n_genes}")
print()
print(f"set par_guide_lm_model = \"OLS\" for notebook 12 to read these fits")